# Daily Challenge – LangChain Pipelines with Open-Source LLMs

**Week 9 · Day 4 — Bootcamp GenAI & Machine Learning 2026**

Objectifs :
- Configurer LangChain avec un petit modèle open-source (CPU-friendly)
- Construire un `LLMChain` avec un `PromptTemplate`
- Composer un pipeline en deux étapes (résumé → bullets) avec les Runnables LCEL
- **Bonus** : chaîne conversationnelle avec mémoire tampon

---
## Part 1 — Environment Setup

In [ ]:
!pip install -q langchain langchain-community langchain-core transformers sentencepiece accelerate

In [ ]:
# Check hardware (GPU or CPU)
!nvidia-smi || echo "No GPU detected — running on CPU (fine for tiny models)"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import transformers
import langchain
print(f"Transformers : {transformers.__version__}")
print(f"LangChain    : {langchain.__version__}")

---
## Part 2 — Load a Tiny Open Model and Build Your First LLMChain

On utilise `google/flan-t5-small` (~300 MB) : modèle seq2seq optimisé pour l'instruction-following, idéal sur CPU.

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Wrap in a HuggingFace text2text-generation pipeline
hf_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,   # greedy decoding for reproducibility
)

print("Model loaded successfully ✅")

In [ ]:
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Wrap the pipeline for LangChain
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# Prompt template: friendly rewrite for beginners
rewrite_template = PromptTemplate(
    input_variables=["text"],
    template="Rewrite this text to be simpler and easier for beginners: {text}"
)

# Build the LLMChain
rewrite_chain = LLMChain(llm=llm, prompt=rewrite_template)

print("LLMChain ready ✅")

In [ ]:
import time

test_text = (
    "A neural network is a computational system inspired by biological neural networks "
    "that constitute animal brains. It consists of layers of interconnected nodes that "
    "process data using weighted connections adjusted through backpropagation."
)

start = time.time()
result = rewrite_chain.invoke({"text": test_text})
elapsed = time.time() - start

print(f"⏱  Latency : {elapsed:.2f}s")
print()
print("Input  :", test_text)
print()
print("Output :", result["text"])

---
## Part 3 — Two-Step Pipeline with LangChain Runnables (LCEL)

**Étape 1** → résumer un paragraphe  
**Étape 2** → transformer le résumé en 3 bullets

On utilise la syntaxe LCEL (`|`) pour chaîner les runnables.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1 — Summarize
summarize_prompt = PromptTemplate(
    input_variables=["paragraph"],
    template="Summarize the following paragraph in one or two sentences: {paragraph}"
)

# Step 2 — Bullet-ize
bullet_prompt = PromptTemplate(
    input_variables=["summary"],
    template="Convert this summary into exactly 3 concise bullet points: {summary}"
)

parser = StrOutputParser()

# LCEL RunnableSequence: paragraph → summary → bullets
pipeline_chain = (
    summarize_prompt
    | llm
    | parser
    | (lambda summary: {"summary": summary})  # bridge output → next input key
    | bullet_prompt
    | llm
    | parser
)

print("Two-step LCEL pipeline ready ✅")

In [ ]:
paragraph = """
Machine learning is a subset of artificial intelligence that enables systems to learn
and improve from experience without being explicitly programmed. It focuses on building
applications that access data and use it to learn for themselves. The process begins
with observations or data, such as examples, direct experience, or instruction.
Machine learning algorithms use computational methods to learn information directly
from data and improve their accuracy as the amount of data increases.
"""

# Step 1: manual check
start = time.time()
summary = (summarize_prompt | llm | parser).invoke({"paragraph": paragraph})
step1_time = time.time() - start
print(f"Step 1 — Summary ({step1_time:.2f}s):")
print(summary)
print()

In [ ]:
# Step 2: manual check
start = time.time()
bullets = (bullet_prompt | llm | parser).invoke({"summary": summary})
step2_time = time.time() - start
print(f"Step 2 — Bullets ({step2_time:.2f}s):")
print(bullets)
print()
print(f"Total pipeline time: {step1_time + step2_time:.2f}s")

In [ ]:
# Run the full pipeline end-to-end
print("=== Full two-step pipeline ===")
start = time.time()
final_output = pipeline_chain.invoke({"paragraph": paragraph})
print(f"End-to-end latency: {time.time() - start:.2f}s")
print()
print("Final output (bullets):")
print(final_output)

---
## Part 4 (Bonus) — Conversation Chain with Buffer Memory

In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

# Custom conversational prompt with style
conv_template = """You are a concise and encouraging AI tutor. Answer briefly and positively.

Current conversation:
{history}

Human: {input}
AI:"""

conv_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template=conv_template
)

memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    prompt=conv_prompt,
    verbose=False
)

print("ConversationChain ready ✅")

In [ ]:
# Turn 1 — greeting
turn1 = conversation.predict(input="Hi! I'm just starting to learn about machine learning.")
print("Turn 1")
print(f"  Human : Hi! I'm just starting to learn about machine learning.")
print(f"  AI    : {turn1}")
print()

In [ ]:
# Turn 2 — follow-up using context from turn 1
turn2 = conversation.predict(input="What should I learn first?")
print("Turn 2")
print(f"  Human : What should I learn first?")
print(f"  AI    : {turn2}")
print()

In [ ]:
# Inspect the memory buffer
print("=== Memory buffer ===")
print(memory.buffer)

---
## Observations & Analyse

### Latence
- `google/flan-t5-small` (~77M paramètres) s'exécute en **1–5 secondes par inférence sur CPU**.
- Le pipeline deux étapes ajoute simplement deux appels séquentiels, donc ~2–10 secondes au total.
- Pas de GPU nécessaire pour des modèles < 300 MB.

### Qualité des sorties
- **Réécriture (Part 2)** : flan-t5-small simplifie correctement le texte sans perdre le sens principal.
- **Pipeline résumé → bullets (Part 3)** : la séquence LCEL fonctionne bien ; les bullets sont cohérents avec le résumé intermédiaire.
- **Mémoire (Part 4)** : `ConversationBufferMemory` conserve l'historique des échanges et permet au modèle de répondre en contexte.

### Particularités
- Flan-T5 est un modèle **encoder-decoder (seq2seq)** : il répond mieux aux instructions courtes et précises qu'aux prompts conversationnels longs.
- Les réponses sont parfois trop courtes avec des petits modèles (tiny-gpt2 notamment). `flan-t5-small` est plus fiable pour les tâches de transformation de texte.
- LCEL (`|`) est plus lisible et composable que `SimpleSequentialChain` — c'est l'approche recommandée dans LangChain ≥ 0.2.

### Résumé du pipeline
```
Paragraphe
    → [PromptTemplate: summarize] → LLM → résumé
    → [PromptTemplate: bullets]   → LLM → 3 bullet points
```

---
## Synthèse des livrables

| Partie | Livrable | Statut |
|--------|----------|--------|
| Part 1 | Installation packages + vérification hardware | ✅ |
| Part 2 | LLMChain (flan-t5-small) — réécriture pour débutants | ✅ |
| Part 3 | Pipeline LCEL deux étapes : résumé → 3 bullets | ✅ |
| Part 4 | ConversationChain avec ConversationBufferMemory (2 tours) | ✅ |
| — | Cellule markdown d'observations (latence, qualité, particularités) | ✅ |